# Monte Carlo Option Pricing

このNotebookでは、ヨーロピアン・オプションのpayoff、Black-Scholes価格、Monte Carlo価格を比較します。

FinSimLabは教育目的のライブラリです。ここで扱う内容は投資助言ではありません。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from finsimlab.options import (
    black_scholes_call,
    black_scholes_delta,
    black_scholes_put,
    call_payoff,
    monte_carlo_option_price,
    put_payoff,
)

spot = 100
strike = 100
time_to_maturity = 1
risk_free_rate = 0.05
volatility = 0.2

## Payoff（満期時の損益）の形

オプションとは、将来の特定の時点で、あらかじめ決めた価格（権利行使価格）で「買う権利」または「売る権利」のことです。

### 例え：予約チケットと保険
- **コール・オプション（買う権利）**は、**「限定商品の予約チケット」**のようなものです。商品の価格が予約価格より高くなれば、安く買える権利には価値が出ます。逆に安くなれば、予約を捨てればいいだけです。
- **プット・オプション（売る権利）**は、**「損害保険」**に似ています。持っている株の価格が暴落しても、あらかじめ決めた価格で売れる（＝守ってもらえる）ため、価格が下がるほどこの権利の価値は高まります。

In [ ]:
terminal_prices = np.linspace(60, 140, 200)

plt.figure(figsize=(8, 4))
plt.plot(terminal_prices, call_payoff(terminal_prices, strike=strike), label="call payoff")
plt.plot(terminal_prices, put_payoff(terminal_prices, strike=strike), label="put payoff")
plt.axvline(strike, color="gray", linestyle="--", linewidth=1, label="strike")
plt.xlabel("Terminal price")
plt.ylabel("Payoff")
plt.legend()
plt.show()

## Black-Scholes価格（理論上の「公式」）

Black-Scholes式は、オプションの適正価格を導き出すための数学的な**「魔法の公式」**です。

### 例え：料理のレシピと完成予想図
材料（株価、ボラティリティ、時間など）を公式に入れると、即座に「理論的な完成品の価格」が出てきます。非常に高速で便利ですが、市場が特殊な状況（レシピの想定外）になると、現実とズレることがあります。

In [ ]:
call_price = black_scholes_call(
    spot=spot,
    strike=strike,
    time_to_maturity=time_to_maturity,
    risk_free_rate=risk_free_rate,
    volatility=volatility,
)
put_price = black_scholes_put(
    spot=spot,
    strike=strike,
    time_to_maturity=time_to_maturity,
    risk_free_rate=risk_free_rate,
    volatility=volatility,
)
delta = black_scholes_delta(
    spot=spot,
    strike=strike,
    time_to_maturity=time_to_maturity,
    risk_free_rate=risk_free_rate,
    volatility=volatility,
    option_type="call",
)

print(f"Call price: {call_price:.4f}")
print(f"Put price:  {put_price:.4f}")
print(f"Call delta: {delta:.4f}")

## Monte Carlo価格（シミュレーションによる「実験」）

モンテカルロ法は、公式を使わずに**「何度も実験を繰り返して平均を取る」**泥臭いけれど強力な方法です。

### 例え：サイコロを何万回も振る実験
「将来の株価がどうなるか」というサイコロを何万回も振り、それぞれの結果でいくら儲かるか（Payoff）を計算します。そのすべての平均を取れば、それがオプションの価値になります。

### 実務上のポイント
Black-Scholes公式が使えないような、複雑な条件（例：途中の価格で価値が変わるオプションなど）を評価する際に、実務ではこのモンテカルロ法が非常に重宝されます。

In [ ]:
for n_paths in [1_000, 10_000, 100_000]:
    estimate = monte_carlo_option_price(
        spot=spot,
        strike=strike,
        time_to_maturity=time_to_maturity,
        risk_free_rate=risk_free_rate,
        volatility=volatility,
        option_type="call",
        n_paths=n_paths,
        seed=42,
    )
    print(f"n_paths={n_paths:>6}: {estimate:.4f}")

print(f"Black-Scholes: {call_price:.4f}")

## 学習メモ: 直感・数式・演習

### 直感

オプションは「将来ある価格で売買できる権利」です。コールオプションは満期価格が権利行使価格を上回るほど価値が出ます。プットオプションはその逆で、満期価格が権利行使価格を下回るほど価値が出ます。

Black-Scholes価格は、一定の仮定のもとで理論価格を直接計算する方法です。Monte Carlo価格は、将来価格を大量にシミュレーションして、満期payoffの平均を現在価値に割り引く方法です。経路数が少ないと推定値はぶれますが、経路数を増やすと理論価格に近づきやすくなります。

### 数式の最小説明

コールとプットのpayoffは次の形です。

$$\text{Call payoff} = \max(S_T - K, 0)$$

$$\text{Put payoff} = \max(K - S_T, 0)$$

Monte Carlo価格は、シミュレーションしたpayoffの平均をリスクフリーレートで割り引きます。

$$\text{Price} \approx e^{-rT} \frac{1}{N}\sum_{i=1}^{N} \text{payoff}^{(i)}$$

ここで、$S_T$ は満期価格、$K$ は権利行使価格、$r$ はリスクフリーレート、$T$ は満期までの年数、$N$ はシミュレーション経路数です。

### 演習問題

1. `strike` を `90`, `100`, `110` に変えて、コール価格とプット価格がどう変わるか比較してください。
2. `volatility` を大きくすると、オプション価格がどう変わるか観察してください。理由も短くメモしてください。
3. `n_paths` を `1_000`, `10_000`, `100_000` に変えて、Monte Carlo推定値の安定性を確認してください。
4. `option_type="put"` に変えて、Monte Carlo価格とBlack-Scholesプット価格を比較してください。

> 注意: オプション価格モデルは仮定に依存します。このNotebookは教育目的であり、取引判断を勧めるものではありません。
